# Big Data Modelling and Management - Lab 2

**Prepare for the class:**

0. download Dataset from Moodle Lab 2 + img folder 
1. run Jupyter 
2. run Neo4j from cmd or as Windows service
3. open http://localhost:7474/browser/  
4. log-in with login: neo4j, password: neo4j [yourpassword]
5. start exploring the database

**2. We will be using the same database we used in Lab 1:**

In [1]:
from neo4j import GraphDatabase
from pprint import pprint

## Define a Connection to your external Database

In [2]:
NEO4J_URI="neo4j://localhost:7687"
NEO4J_USERNAME="neo4j"
NEO4J_PASSWORD="Neo4jneo4j"

In [3]:
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD), )

## All The Functions you'll need to run queries in Neo4J

In [4]:
def execute_write(driver, query):
    with driver.session(database="neo4j") as session:
        # Write transactions allow the driver to handle retries and transient errors
        result = session.execute_write(lambda tx, query: list(tx.run(query)), query)
    return result

In [5]:
def execute_read(driver, query):    
    with driver.session(database="neo4j") as session:
        result = session.execute_read(lambda tx, query: list(tx.run(query)), query)
    return result

In [6]:
query = """
        MATCH () RETURN count(*)
    """

result = execute_read(driver, query)
pprint(result)

[<Record count(*)=179>]


## Today we will answer some questions about our database

In [7]:
query = """
        call db.labels;
    """

result = execute_read(driver, query)
pprint(result)

[<Record label='Movie'>,
 <Record label='Person'>,
 <Record label='User'>,
 <Record label='Course'>,
 <Record label='Student'>,
 <Record label='Tweet'>]


In [8]:
query = """
        MATCH (node) 
        RETURN distinct labels(node)
    """

result = execute_read(driver, query)

pprint(result)

[<Record labels(node)=['Movie']>,
 <Record labels(node)=['Person']>,
 <Record labels(node)=['User']>,
 <Record labels(node)=['Course']>,
 <Record labels(node)=['User', 'Student']>,
 <Record labels(node)=['Tweet']>]


## Counting

### List the actors based on their movie count from max to min

In [9]:
query = """
        MATCH (m:Movie)-[r:ACTED_IN]-(p:Person)
        RETURN p.name, count(r) as MovieCount
        ORDER BY count(r) DESC
        LIMIT 5
    """
result = execute_read(driver, query)
pprint(result)

[<Record p.name='Tom Hanks' MovieCount=12>,
 <Record p.name='Keanu Reeves' MovieCount=7>,
 <Record p.name='Jack Nicholson' MovieCount=5>,
 <Record p.name='Hugo Weaving' MovieCount=5>,
 <Record p.name='Meg Ryan' MovieCount=5>]


In [10]:
query = """
        MATCH ()-[]-(m:Movie)
        RETURN count(m)
    """

result = execute_read(driver, query)

pprint(result)

[<Record count(m)=250>]


In [11]:
# How many people are participated in any way to a specific movie
query = """
        MATCH (m:Movie {title: "The Replacements"})<-[r]-(p:Person)
        RETURN count(r)
    """
result = execute_read(driver, query)
pprint(result)

[<Record count(r)=8>]


### List movies with more than one director

In [12]:
query = """
        MATCH path = (p1:Person)-[:DIRECTED]-(m:Movie)-[:DIRECTED]-(p2:Person)
        RETURN p1.name, m.title, p2.name, path
    """
result = execute_read(driver, query)
pprint(result)

[<Record p1.name='Lilly Wachowski' m.title='The Matrix' p2.name='Lana Wachowski' path=<Path start=<Node element_id='4:3b43e673-3b97-4b3f-93e5-776eac49a647:5' labels=frozenset({'Person'}) properties={'born': 1967, 'name': 'Lilly Wachowski'}> end=<Node element_id='4:3b43e673-3b97-4b3f-93e5-776eac49a647:6' labels=frozenset({'Person'}) properties={'born': 1965, 'name': 'Lana Wachowski'}> size=2>>,
 <Record p1.name='Lana Wachowski' m.title='The Matrix' p2.name='Lilly Wachowski' path=<Path start=<Node element_id='4:3b43e673-3b97-4b3f-93e5-776eac49a647:6' labels=frozenset({'Person'}) properties={'born': 1965, 'name': 'Lana Wachowski'}> end=<Node element_id='4:3b43e673-3b97-4b3f-93e5-776eac49a647:5' labels=frozenset({'Person'}) properties={'born': 1967, 'name': 'Lilly Wachowski'}> size=2>>,
 <Record p1.name='Lilly Wachowski' m.title='The Matrix Reloaded' p2.name='Lana Wachowski' path=<Path start=<Node element_id='4:3b43e673-3b97-4b3f-93e5-776eac49a647:5' labels=frozenset({'Person'}) properties

In [13]:
query = """
        MATCH path = (p1:Person)-[:DIRECTED]-(m:Movie)-[:DIRECTED]-(p2:Person)
        RETURN count(path) // as count, path
    """
result = execute_read(driver, query)
pprint(result)

[<Record count(path)=14>]


In [14]:
query = """
        MATCH path = (p1:Person)-[:DIRECTED]-(m:Movie)-[:DIRECTED]-(p2:Person)
        RETURN  distinct [p1.name,  p2.name] //count(distinct [p1.name, p2.name])
    """
result = execute_read(driver, query)
pprint(result)

[<Record [p1.name,  p2.name]=['Lilly Wachowski', 'Lana Wachowski']>,
 <Record [p1.name,  p2.name]=['Lana Wachowski', 'Lilly Wachowski']>,
 <Record [p1.name,  p2.name]=['Tom Tykwer', 'Lana Wachowski']>,
 <Record [p1.name,  p2.name]=['Tom Tykwer', 'Lilly Wachowski']>,
 <Record [p1.name,  p2.name]=['Lana Wachowski', 'Tom Tykwer']>,
 <Record [p1.name,  p2.name]=['Lilly Wachowski', 'Tom Tykwer']>]


In [15]:
# Using UNWIND which takes a list and turns each element into a separate row.

query = """
        MATCH (p1:Person)-[:DIRECTED]-(m:Movie)-[:DIRECTED]-(p2:Person)
        
        UNWIND [p1, p2] AS director
        
        RETURN count(DISTINCT director) AS totalDirectors
    """
result = execute_read(driver, query)
pprint(result)

[<Record totalDirectors=3>]


### Find all movies of the actors who also directed movies 

In [16]:
query = """
        MATCH (m:Movie)-[r1:ACTED_IN]-(p:Person)-[r2:DIRECTED]-(m1:Movie)
        RETURN * 
    """

result = execute_read(driver, query)
pprint(result)

[<Record m=<Node element_id='4:3b43e673-3b97-4b3f-93e5-776eac49a647:162' labels=frozenset({'Movie'}) properties={'tagline': 'Once in a lifetime you get a chance to do something different.', 'title': 'A League of Their Own', 'released': 1992}> m1=<Node element_id='4:3b43e673-3b97-4b3f-93e5-776eac49a647:85' labels=frozenset({'Movie'}) properties={'tagline': 'In every life there comes a time when that thing you dream becomes that thing you do', 'title': 'That Thing You Do', 'released': 1996}> p=<Node element_id='4:3b43e673-3b97-4b3f-93e5-776eac49a647:68' labels=frozenset({'Person'}) properties={'born': 1956, 'name': 'Tom Hanks'}> r1=<Relationship element_id='5:3b43e673-3b97-4b3f-93e5-776eac49a647:234' nodes=(<Node element_id='4:3b43e673-3b97-4b3f-93e5-776eac49a647:68' labels=frozenset({'Person'}) properties={'born': 1956, 'name': 'Tom Hanks'}>, <Node element_id='4:3b43e673-3b97-4b3f-93e5-776eac49a647:162' labels=frozenset({'Movie'}) properties={'tagline': 'Once in a lifetime you get a cha

In [17]:
# List all actors who also directed movies

query = """
        MATCH (m:Movie)-[r1:ACTED_IN]-(p:Person), 
              (n)-[r2:DIRECTED]-(p) 
        RETURN   p.name //distinct
    """
result = execute_read(driver, query)
pprint(result)

[<Record p.name='Tom Hanks'>,
 <Record p.name='Tom Hanks'>,
 <Record p.name='Tom Hanks'>,
 <Record p.name='Tom Hanks'>,
 <Record p.name='Tom Hanks'>,
 <Record p.name='Tom Hanks'>,
 <Record p.name='Tom Hanks'>,
 <Record p.name='Tom Hanks'>,
 <Record p.name='Tom Hanks'>,
 <Record p.name='Tom Hanks'>,
 <Record p.name='Tom Hanks'>,
 <Record p.name='Tom Hanks'>,
 <Record p.name='Werner Herzog'>,
 <Record p.name='Clint Eastwood'>,
 <Record p.name='James Marshall'>,
 <Record p.name='James Marshall'>,
 <Record p.name='Danny DeVito'>,
 <Record p.name='Danny DeVito'>]


In [18]:
query = """
        MATCH (m:Movie)-[r1:ACTED_IN]-(p:Person), 
              (n)-[r2:DIRECTED]-(p) 
        RETURN * 
    """

result = execute_read(driver, query)
pprint(result)

[<Record m=<Node element_id='4:3b43e673-3b97-4b3f-93e5-776eac49a647:162' labels=frozenset({'Movie'}) properties={'tagline': 'Once in a lifetime you get a chance to do something different.', 'title': 'A League of Their Own', 'released': 1992}> n=<Node element_id='4:3b43e673-3b97-4b3f-93e5-776eac49a647:85' labels=frozenset({'Movie'}) properties={'tagline': 'In every life there comes a time when that thing you dream becomes that thing you do', 'title': 'That Thing You Do', 'released': 1996}> p=<Node element_id='4:3b43e673-3b97-4b3f-93e5-776eac49a647:68' labels=frozenset({'Person'}) properties={'born': 1956, 'name': 'Tom Hanks'}> r1=<Relationship element_id='5:3b43e673-3b97-4b3f-93e5-776eac49a647:234' nodes=(<Node element_id='4:3b43e673-3b97-4b3f-93e5-776eac49a647:68' labels=frozenset({'Person'}) properties={'born': 1956, 'name': 'Tom Hanks'}>, <Node element_id='4:3b43e673-3b97-4b3f-93e5-776eac49a647:162' labels=frozenset({'Movie'}) properties={'tagline': 'Once in a lifetime you get a chan

In [19]:
# When a query contains multiple disconnected patterns, this will build a cartesian product between all those parts.

query = """
        MATCH (m:Movie)-[r1:ACTED_IN]-(p1:Person), 
              ()-[r2:REVIEWED]-(p2:Person) 
        RETURN * 
    """
result = execute_read(driver, query)
pprint(result)

[<Record m=<Node element_id='4:3b43e673-3b97-4b3f-93e5-776eac49a647:0' labels=frozenset({'Movie'}) properties={'tagline': 'Welcome to the Real World', 'title': 'The Matrix', 'released': 1999}> p1=<Node element_id='4:3b43e673-3b97-4b3f-93e5-776eac49a647:8' labels=frozenset({'Person'}) properties={'born': 1978, 'name': 'Emil Eifrem'}> p2=<Node element_id='4:3b43e673-3b97-4b3f-93e5-776eac49a647:169' labels=frozenset({'Person'}) properties={'name': 'Jessica Thompson'}> r1=<Relationship element_id='5:3b43e673-3b97-4b3f-93e5-776eac49a647:7' nodes=(<Node element_id='4:3b43e673-3b97-4b3f-93e5-776eac49a647:8' labels=frozenset({'Person'}) properties={'born': 1978, 'name': 'Emil Eifrem'}>, <Node element_id='4:3b43e673-3b97-4b3f-93e5-776eac49a647:0' labels=frozenset({'Movie'}) properties={'tagline': 'Welcome to the Real World', 'title': 'The Matrix', 'released': 1999}>) type='ACTED_IN' properties={'roles': ['Emil']}> r2=<Relationship element_id='5:3b43e673-3b97-4b3f-93e5-776eac49a647:244' nodes=(<

In [20]:
# We can use OPTIONAL MATCH to check a hypothesis.

query = """
        MATCH (m:Movie)-[r1:ACTED_IN]-(p:Person)
        OPTIONAL MATCH ()-[r2:REVIEWED]-(p)
        RETURN * // count(r2)
    """
result = execute_read(driver, query)
pprint(result)
## OPTIONAL MATCH behaves like a left outer join: if no matching review exists, the actor and movie are still returned, with r2 set to null.

[<Record m=<Node element_id='4:3b43e673-3b97-4b3f-93e5-776eac49a647:0' labels=frozenset({'Movie'}) properties={'tagline': 'Welcome to the Real World', 'title': 'The Matrix', 'released': 1999}> p=<Node element_id='4:3b43e673-3b97-4b3f-93e5-776eac49a647:8' labels=frozenset({'Person'}) properties={'born': 1978, 'name': 'Emil Eifrem'}> r1=<Relationship element_id='5:3b43e673-3b97-4b3f-93e5-776eac49a647:7' nodes=(<Node element_id='4:3b43e673-3b97-4b3f-93e5-776eac49a647:8' labels=frozenset({'Person'}) properties={'born': 1978, 'name': 'Emil Eifrem'}>, <Node element_id='4:3b43e673-3b97-4b3f-93e5-776eac49a647:0' labels=frozenset({'Movie'}) properties={'tagline': 'Welcome to the Real World', 'title': 'The Matrix', 'released': 1999}>) type='ACTED_IN' properties={'roles': ['Emil']}> r2=None>,
 <Record m=<Node element_id='4:3b43e673-3b97-4b3f-93e5-776eac49a647:0' labels=frozenset({'Movie'}) properties={'tagline': 'Welcome to the Real World', 'title': 'The Matrix', 'released': 1999}> p=<Node elemen

### Which movies have the most actors involved?

In [21]:
query = """
        MATCH (m:Movie)-[r:ACTED_IN]-(p:Person)
        RETURN m.title, count(r) as ActorCount
        ORDER BY count(r) DESC
        LIMIT 6
    """
result = execute_read(driver, query)
pprint(result)

[<Record m.title='A Few Good Men' ActorCount=12>,
 <Record m.title='Jerry Maguire' ActorCount=9>,
 <Record m.title='The Green Mile' ActorCount=8>,
 <Record m.title='Stand By Me' ActorCount=7>,
 <Record m.title='Speed Racer' ActorCount=7>,
 <Record m.title='Top Gun' ActorCount=6>]


In [22]:
# WHERE operator to add condition on an integer value of a property

query = """
        MATCH (m:Movie)-[r:ACTED_IN]-(p:Person)
        WHERE m.released > 2000 // >,<,=,<>
        RETURN m.title, count(r) AS ActorCount
        ORDER BY ActorCount DESC
        LIMIT 6
    """
result = execute_read(driver, query)
pprint(result)

[<Record m.title='Speed Racer' ActorCount=7>,
 <Record m.title='Frost/Nixon' ActorCount=5>,
 <Record m.title='V for Vendetta' ActorCount=5>,
 <Record m.title='RescueDawn' ActorCount=4>,
 <Record m.title='The Da Vinci Code' ActorCount=4>,
 <Record m.title='Cloud Atlas' ActorCount=4>]


In [23]:
# Two WHERE clauses: on an integer value of a property, and on a string value

query = """
        MATCH (m:Movie)-[r:ACTED_IN]-(p:Person)
        WHERE m.released > 1990
          AND m.tagline STARTS WITH 'A'  // OR
        RETURN m.title, count(r) AS ActorCount, m.tagline
        ORDER BY ActorCount DESC
        LIMIT 6
    """
result = execute_read(driver, query)
pprint(result)

[<Record m.title="You've Got Mail" ActorCount=6 m.tagline='At odds in life... in love on-line.'>,
 <Record m.title='What Dreams May Come' ActorCount=5 m.tagline='After life there is more. The end is just the beginning.'>,
 <Record m.title='As Good as It Gets' ActorCount=4 m.tagline='A comedy from the heart that goes for the throat.'>,
 <Record m.title="Charlie Wilson's War" ActorCount=3 m.tagline="A stiff drink. A little mascara. A lot of nerve. Who said they couldn't bring down the Soviet empire.">,
 <Record m.title='Cast Away' ActorCount=2 m.tagline='At the edge of the world, his journey begins.'>]


### Who is reviewed using specific words?

In [24]:
# Using f-string (formated string) which enables to externaly define.
subject = "funny"

query = f"""
        MATCH (p)-[r]->(m)
        WHERE r.summary contains '{subject}'
        RETURN p.name, r.summary
        LIMIT 5
    """
result = execute_read(driver, query)
pprint(result)

[<Record p.name='Angela Scope' r.summary='Pretty funny at times'>]


In [25]:
subjects = ['Germany', 'France']

query = f"""
        WITH {subjects} AS words

        MATCH (u:User)-[r:TWEETED]->(t:Tweet)
        
        WHERE all(word IN words WHERE t.text CONTAINS word)

        RETURN u.username, t.text
        LIMIT 5
    """

result = execute_read(driver, query)

pprint(result)

[]


In [26]:
# Using operator WITH which creates an intermediate variable (keywords) and make it available to the following Cypher clauses.

words = ['funny', 'Pretty']

query = f"""
        WITH {words} AS keywords

        MATCH (p:Person)-[r:REVIEWED]->(m:Movie)
        
        WHERE all(item IN keywords WHERE r.summary CONTAINS item) //any

        RETURN p.name, r.summary
    """
result = execute_read(driver, query)
pprint(result)

[<Record p.name='Angela Scope' r.summary='Pretty funny at times'>]


In [27]:
query = """
        MATCH path = (p1:Person)-[:DIRECTED]-(m:Movie)-[:DIRECTED]-(p2:Person)
        RETURN m.title //count(path) as count, path 
    """
result = execute_read(driver, query)
pprint(result)

[<Record m.title='The Matrix'>,
 <Record m.title='The Matrix'>,
 <Record m.title='The Matrix Reloaded'>,
 <Record m.title='The Matrix Reloaded'>,
 <Record m.title='The Matrix Revolutions'>,
 <Record m.title='The Matrix Revolutions'>,
 <Record m.title='Cloud Atlas'>,
 <Record m.title='Cloud Atlas'>,
 <Record m.title='Cloud Atlas'>,
 <Record m.title='Cloud Atlas'>,
 <Record m.title='Cloud Atlas'>,
 <Record m.title='Cloud Atlas'>,
 <Record m.title='Speed Racer'>,
 <Record m.title='Speed Racer'>]


In [28]:
query = """
        MATCH path = (p1:Person)-[:DIRECTED]-(m:Movie)-[:DIRECTED]-(p2:Person)
        RETURN collect(m.title)         
    """
result = execute_read(driver, query)
pprint(result)

[<Record collect(m.title)=['Cloud Atlas', 'Cloud Atlas', 'Cloud Atlas', 'Cloud Atlas', 'Cloud Atlas', 'Cloud Atlas', 'Speed Racer', 'Speed Racer', 'The Matrix', 'The Matrix', 'The Matrix Reloaded', 'The Matrix Reloaded', 'The Matrix Revolutions', 'The Matrix Revolutions']>]
